# Part 3: Lead time

The target stays at **12:00 on day d+1**; only the issue time moves closer:

| lead | issue time |
|---|---|
| 24 h | 12:00 on day d (the main task) |
| 12 h | 00:00 on d+1 |
| 6 h | 06:00 on d+1 |
| 3 h | 09:00 on d+1 |
| 1 h | 11:00 on d+1 |

For every lead: (1) persistence = the reading at the issue time; (2) the Part 1 ridge with the same 29 features taken relative to the new issue time, refitted for that lead; (3) our Part 2 model (Sanath's Laplace linear model on `log1p`, smeared mean, `cache/features_C.py`), refitted for that lead with the same hyper-parameters. All features are computed from readings at or before the issue time, so the same code works for every lead. Training targets are 3 Jan – 31 Aug (113,256 rows at every lead) and validation targets are 1 Sep – 30 Nov (42,588 rows at every lead), so all leads are scored on exactly the same targets.

In [ ]:
import os, sys, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd()
if not (ROOT / "baseline.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import baseline as B
from cache import features_C as FC
from cache import probmodels_C as PM
B.self_check()

W, stations = B.load_data()
ctx = FC.build_ctx(W, stations)        # neighbour matrices (fitted quantities use training hours only)
LEADS = [1, 3, 6, 12, 24]

## 1. Fit and score every method at every lead

In [ ]:
rows, preds = [], {}
for lead in LEADS:
    T_train, T_val = B.issue_times(lead)
    y_train, y_val = B.targets(W, T_train, lead), B.targets(W, T_val, lead)
    assert len(y_train) == 113_256 and len(y_val) == 42_588

    # leakage check at this lead: features from history cut at t are identical
    t = T_val[10]
    a, _, _ = FC.extract_features(W, [t], ctx)
    b, _, _ = FC.extract_features(W.loc[:t], [t], ctx)
    assert np.allclose(a, b), f"leakage at lead {lead}"

    X_train, _, _ = FC.extract_features(W, T_train, ctx)
    X_val, _, _ = FC.extract_features(W, T_val, ctx)
    laplace = PM.LogLaplaceForecaster().fit(X_train, y_train)

    p = {"Persistence": B.persistence(W, T_val),
         "Ridge (Part 1, refitted)": B.part1_ridge_predict(W, stations, T_val, lead),
         "Laplace (Part 2, refitted)": laplace.predict(X_val)}
    preds[lead] = p
    for method, pred in p.items():
        e = B.evaluate(pred, y_val, T_val, lead)
        rows.append({"lead (h)": lead, "issue time": f"{T_val[0]:%H:%M}" + (" (day d)" if lead == 24 else " (day d+1)"),
                     "method": method, "RMSE": e["RMSE"], "MAE": e["MAE"],
                     "RMSE Sep": e["RMSE Sep"], "RMSE Oct": e["RMSE Oct"], "RMSE Nov": e["RMSE Nov"],
                     "bias": float(np.mean(np.clip(pred, 0, None) - y_val))})
    print(f"lead {lead:2d} h done (issue {T_val[0]:%H:%M}, smearing factor {laplace.smearing_:.3f})")

results = pd.DataFrame(rows)
results.to_csv("part3_results.csv", index=False)
results.pivot(index="lead (h)", columns="method", values="RMSE").round(2)

## 2. Validation RMSE against lead time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    for method, style in [("Persistence", "o--"), ("Ridge (Part 1, refitted)", "s-"), ("Laplace (Part 2, refitted)", "^-")]:
        d = results[results.method == method]
        ax.plot(d["lead (h)"], d[metric], style, label=method)
    ax.set_xticks(LEADS); ax.set_xlabel("lead time (h); target fixed at 12:00 on day d+1")
    ax.set_ylabel(f"validation {metric} (µg/m³)"); ax.set_title(f"Validation {metric} vs lead time")
    ax.grid(alpha=0.3)
axes[0].legend()
fig.tight_layout(); fig.savefig("part3_rmse_vs_lead.png", dpi=150); plt.show()

## 3. The 1-hour nowcast and the day-ahead forecast, reported separately

In [ ]:
cols = ["method", "RMSE", "MAE", "RMSE Sep", "RMSE Oct", "RMSE Nov", "bias"]
print("1-hour lead (nowcast, issued 11:00 on day d+1)")
display(results[results["lead (h)"] == 1][cols].set_index("method").round(2))
print("24-hour lead (day-ahead, issued 12:00 on day d)")
display(results[results["lead (h)"] == 24][cols].set_index("method").round(2))

## 4. Why does persistence behave so differently across leads?
Average PM2.5 by hour of day (IST) over all stations, for the training and validation periods. The issue hours used above are marked.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
for label, D in [("training (Jan–Aug)", W.loc[:"2024-08-31"]), ("validation (Sep–Nov)", W.loc["2024-09-01":])]:
    by_hour = D.groupby(D.index.hour).mean().mean(axis=1)
    ax.plot(by_hour.index, by_hour.values, marker="o", label=label)
for lead, hour in [(24, 12), (12, 0), (6, 6), (3, 9), (1, 11)]:
    ax.axvline(hour, color="grey", lw=0.8, ls=":")
    ax.text(hour + 0.2, ax.get_ylim()[1] * 0.97, f"{lead} h", fontsize=8, va="top")
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("hour of day (IST)"); ax.set_ylabel("mean PM2.5 (µg/m³)")
ax.set_title("Average daily cycle of PM2.5 (target hour = 12:00)"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("part3_daily_cycle.png", dpi=150); plt.show()

V = W.loc["2024-09-01":]
cycle = V.groupby(V.index.hour).mean().mean(axis=1)
pd.DataFrame({"issue hour": [12, 0, 6, 9, 11], "lead (h)": [24, 12, 6, 3, 1],
              "mean PM2.5 at issue hour (val)": [cycle[h] for h in [12, 0, 6, 9, 11]],
              "mean PM2.5 at 12:00 (val)": cycle[12],
              "persistence bias": [results.query("method == 'Persistence' and `lead (h)` == @l")["bias"].item() for l in [24, 12, 6, 3, 1]]}
             ).round(1)

## 5. Answers

**Does persistence behave differently at 1 h, 24 h and in between? Why?**
Yes, and its error is not monotonic in lead time. Validation RMSE is 23.95 at 1 h, then 51.43 at 3 h, 66.82 at 6 h and 72.74 at 12 h, but it drops back to 32.75 at 24 h. The reason is the daily cycle of PM2.5 (section 4). In the validation period the network average is about 93 µg/m³ at midnight and about 88 at 06:00, but only about 53 at noon: at night the boundary layer is shallow and pollution is trapped near the ground, while daytime heating mixes it through a deeper layer. Persistence copies the reading at the issue hour, so from 00:00 or 06:00 it carries the night-time level into a noon target and over-predicts by about 40 and 35 µg/m³ on average (the bias column). At 24 h the issue hour and the target hour are both 12:00, so the daily cycle cancels and only the genuine day-to-day change remains. That is the "dip" at 24 h. At 1 h the issue hour (11:00) is already close to the noon level, so persistence is good.

**Do the learned models show the same behaviour?**
Mostly not, because each is refitted for its lead and learns the average night-to-noon relation. The Laplace model's RMSE is 20.58 at 1 h and then nearly flat, 27.25 to 29.08 for 3 to 24 h, and the Part 1 ridge rises from 18.95 to a maximum of 33.72 at 6 h and comes back to 29.58 at 24 h. A small dip at 24 h remains for both: from noon on day d the model has today's noon reading, the same hour as the target, while from 00:00 or 06:00 it has to infer how strongly the next morning will mix. The log-space Laplace model handles this much better than the level-space ridge (6 h: 29.08 vs 33.72), because the night-to-noon change is roughly a fixed *proportion* (noon is about 60% of the 00:00 level in the validation median), which is a constant in log space but level-dependent in µg/m³. At 1 h the ridge is better (18.95 vs 20.58): when the target is almost the current reading, an additive model fitted directly on squared error is the natural fit, while the Laplace model over-predicts slightly at this lead (bias +4.2 against +0.5 for ridge); with so little left to predict, a bias of a few µg/m³ is a large part of the error.

**Which lead time for a public warning service, and what does it cost?**
The 24-hour lead (issued at 12:00 on day d). A warning is only useful if people can act on it: plan outdoor activities, schools or work, and the protective steps they need, the day before. It costs surprisingly little accuracy: our model's RMSE is 28.16 at 24 h against 20.58 at 1 h (7.6 µg/m³ worse), and it is no worse than issuing at midnight or 06:00 (28.63 and 29.08), which would give less warning. A 1-hour nowcast is more accurate but leaves too little time to act; it is useful as a same-day update, not as the advisory.